In [1]:
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
from llm import llm

In [2]:
class Joke(TypedDict):
    topic:str
    joke:str
    explain:str

In [21]:
def write(state:Joke):
    res=llm.invoke(f"Write a dark , darkest joke on topic - {state['topic']}").content
    return {'joke':res}
def exp(state:Joke):
    res=llm.invoke(f"Explain the joke - {state['joke']}").content
    return {'explain':res}

In [22]:
graph=StateGraph(Joke)
checkpoint=InMemorySaver()

graph.add_node("write",write)
graph.add_node("exp",exp)

graph.add_edge(START,"write")
graph.add_edge("write","exp")
graph.add_edge("exp",END)

wf=graph.compile(checkpointer=checkpoint)


In [23]:
config={"configurable":{"thread_id":'1'}}
res=wf.invoke({"topic":"Introvert"},config=config)

In [26]:
list(wf.get_state_history(config)) 

[StateSnapshot(values={'topic': 'Introvert', 'joke': "Why did the introvert's cat join a support group?\n\nBecause it was tired of being the only one who didn't judge the introvert for having been dead inside for years, and the cat was starting to think it was just a normal part of the furniture.", 'explain': 'A joke that pokes fun at introverts and their sometimes bleak outlook on life. Let\'s break it down:\n\nThe joke starts by setting up the idea that an introvert\'s cat has joined a support group. This is already unusual, as cats don\'t typically join support groups. The punchline explains that the cat joined the group because it was tired of being the only one who didn\'t judge the introvert for being "dead inside" for years.\n\nThe phrase "dead inside" is a humorous exaggeration of the introvert\'s emotional state. It\'s a common stereotype that introverts are aloof, reserved, and possibly emotionally numb. The joke takes this stereotype to an absurd extreme, implying that the i